## Purpose

This notebook turns the parametric-estimation ideas from the Yandex ML Handbook article [Parametric estimates](https://education.yandex.ru/handbook/ml/article/parametricheskie-ocenki) into a compact mathematical and computational tour.

We will state the Law of Large Numbers and the Central Limit Theorem, visualize both, define key estimator properties, derive bias-variance decomposition, compare estimator efficiency, derive maximum likelihood estimators, and finally treat logistic regression as Bernoulli maximum likelihood on public Bybit candle data.

> The final Bybit example is for statistical modeling practice only. It is not financial advice and not a trading strategy.

In [1]:
from __future__ import annotations

import time
from typing import Any

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from IPython.display import HTML
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
RNG = np.random.default_rng(RANDOM_STATE)

pd.options.display.float_format = "{:.4f}".format
px.defaults.template = "plotly_white"


def show_plot(fig: go.Figure, height: int = 500) -> HTML:
    """Render Plotly as saved HTML so Quarto can publish the output."""
    fig.update_layout(
        height=height,
        margin={"l": 40, "r": 30, "t": 70, "b": 40},
    )
    return HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

## 1. Limit theorems

Let $X_1, X_2, \ldots$ be independent identically distributed random variables with mean $\mu = E[X_1]$.

**Law of Large Numbers (LLN).** If $E|X_1| < \infty$, then the sample mean

$$
\overline X_n = \frac{1}{n}\sum_{i=1}^{n} X_i
$$

converges to $\mu$ in probability:

$$
\overline X_n \xrightarrow{P} \mu.
$$

A stronger version says that, under standard iid integrability assumptions, $\overline X_n \to \mu$ almost surely.

The theorem explains why averages become stable: the randomness in individual observations does not vanish, but averaging cancels it.

In [2]:
mu = 1.0
sigma = 2.0
n_observations = 2_000
n_paths = 25

samples = RNG.normal(loc=mu, scale=sigma, size=(n_paths, n_observations))
running_means = np.cumsum(samples, axis=1) / np.arange(1, n_observations + 1)

fig = go.Figure()
x_values = np.arange(1, n_observations + 1)
for path_id, path in enumerate(running_means, start=1):
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=path,
            mode="lines",
            line={"width": 1},
            opacity=0.45,
            name=f"path {path_id}",
            showlegend=False,
            hovertemplate="n=%{x}<br>sample mean=%{y:.3f}<extra></extra>",
        )
    )

fig.add_trace(
    go.Scatter(
        x=[1, n_observations],
        y=[mu, mu],
        mode="lines",
        line={"color": "black", "dash": "dash", "width": 3},
        name="true mean",
        hovertemplate="true mean=%{y:.3f}<extra></extra>",
    )
)
fig.update_layout(
    title="Law of Large Numbers: sample means stabilize",
    xaxis_title="sample size n",
    yaxis_title="running sample mean",
    template="plotly_white",
)
show_plot(fig, height=520)

**Central Limit Theorem (CLT).** If $E[X_1]=\mu$ and $0 < Var(X_1)=\sigma^2 < \infty$, then

$$
\frac{\sqrt n(\overline X_n - \mu)}{\sigma}
\xrightarrow{d} N(0,1).
$$

Equivalently, for large $n$,

$$
\overline X_n \approx N\left(\mu, \frac{\sigma^2}{n}\right).
$$

The original observations do not need to be normally distributed. The normal shape appears in the distribution of standardized averages.

In [5]:
n = 400
n_trials = 50_000

# Exponential observations are skewed, so this is a good CLT stress test.
raw = RNG.exponential(scale=1.0, size=(n_trials, n))
sample_means = raw.mean(axis=1)
standardized = np.sqrt(n) * (sample_means - 1.0) / 1.0

grid = np.linspace(-4, 4, 500)
normal_density = stats.norm.pdf(grid)

fig = go.Figure()
fig.add_trace(
    go.Histogram(
        x=standardized,
        histnorm="probability density",
        nbinsx=90,
        name="standardized sample means",
        opacity=0.75,
        hovertemplate="z=%{x:.3f}<br>density=%{y:.3f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=grid,
        y=normal_density,
        mode="lines",
        line={"color": "black", "width": 3},
        name="N(0, 1) density",
        hovertemplate="z=%{x:.3f}<br>density=%{y:.3f}<extra></extra>",
    )
)
fig.update_layout(
    title="Central Limit Theorem from exponential observations",
    xaxis_title="standardized mean",
    yaxis_title="density",
    template="plotly_white",
    barmode="overlay",
)
show_plot(fig, height=520)

## 2. Estimator properties

Let $X_1, \ldots, X_n$ be a sample from a distribution with unknown parameter $\theta$. An **estimator** is a statistic $\widehat\theta_n = T(X_1, \ldots, X_n)$ used to approximate $\theta$.

**Unbiased estimator.** $\widehat\theta_n$ is unbiased for $\theta$ if

$$
E_\theta[\widehat\theta_n] = \theta
$$

for every admissible $\theta$.

**Consistent estimator.** $\widehat\theta_n$ is consistent if

$$
\widehat\theta_n \xrightarrow{P} \theta.
$$

**Asymptotically normal estimator.** $\widehat\theta_n$ is asymptotically normal if there exists a positive sequence $a_n$ and variance $V(\theta)$ such that

$$
a_n(\widehat\theta_n - \theta) \xrightarrow{d} N(0, V(\theta)).
$$

Most regular parametric estimators use $a_n=\sqrt n$.

For the sample mean $\overline X_n$ of iid observations with mean $\mu$ and variance $\sigma^2$:

$$
E[\overline X_n] = \mu, \qquad Var(\overline X_n)=\frac{\sigma^2}{n}.
$$

Thus the sample mean is unbiased. By LLN it is consistent, and by CLT it is asymptotically normal.

In [6]:
true_mu = 3.0
true_sigma = 2.0
sample_sizes = np.array([5, 10, 30, 100, 300, 1_000])
trials = 40_000

rows = []
for n in sample_sizes:
    sample_means = RNG.normal(true_mu, true_sigma, size=(trials, n)).mean(axis=1)
    empirical_bias = sample_means.mean() - true_mu
    empirical_variance = sample_means.var(ddof=1)
    theoretical_variance = true_sigma**2 / n
    rows.append(
        {
            "n": n,
            "empirical_bias": empirical_bias,
            "empirical_variance": empirical_variance,
            "theoretical_variance": theoretical_variance,
        }
    )

mean_properties = pd.DataFrame(rows)
mean_properties

,n,empirical_bias,empirical_variance,theoretical_variance
0,5,0.0005,0.7885,0.8000
1,10,0.0002,0.4007,0.4000
2,30,0.0004,0.1340,0.1333
3,100,0.0020,0.0399,0.0400
4,300,0.0003,0.0134,0.0133
5,1000,0.0005,0.0040,0.0040


In [7]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=mean_properties["n"],
        y=mean_properties["empirical_variance"],
        mode="lines+markers",
        name="empirical variance",
        hovertemplate="n=%{x}<br>variance=%{y:.5f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=mean_properties["n"],
        y=mean_properties["theoretical_variance"],
        mode="lines+markers",
        line={"dash": "dash"},
        name="sigma^2 / n",
        hovertemplate="n=%{x}<br>variance=%{y:.5f}<extra></extra>",
    )
)
fig.update_layout(
    title="Sample mean variance decreases at rate 1/n",
    xaxis_title="sample size n",
    yaxis_title="variance of the estimator",
    template="plotly_white",
    xaxis_type="log",
    yaxis_type="log",
)
show_plot(fig)

Unbiasedness and consistency are different properties. An estimator can be biased but consistent if its bias goes to zero. Conversely, an unbiased estimator need not be consistent if its variance does not shrink.

## 3. Bias-variance decomposition

For an estimator $\widehat\theta$ of a scalar parameter $\theta$, define

$$
Bias(\widehat\theta)=E[\widehat\theta]-\theta.
$$

The mean squared error is

$$
MSE(\widehat\theta)=E[(\widehat\theta-\theta)^2].
$$

Add and subtract $E[\widehat\theta]$:

$$
\widehat\theta-\theta =
(\widehat\theta-E[\widehat\theta]) + (E[\widehat\theta]-\theta).
$$

After squaring and taking expectation, the cross term is zero, so

$$
MSE(\widehat\theta)=Var(\widehat\theta)+Bias(\widehat\theta)^2.
$$

it is one of the main ways to compare estimators.

In [8]:
theta = 5.0
n = 20
trials = 80_000

sample = RNG.normal(loc=theta, scale=2.0, size=(trials, n))
estimator_unbiased = sample.mean(axis=1)
estimator_shrunken = 0.9 * sample.mean(axis=1) + 0.1 * theta
estimator_wrong_shrinkage = 0.9 * sample.mean(axis=1)

def summarize_estimator(values: np.ndarray, target: float, name: str) -> dict[str, float | str]:
    bias = values.mean() - target
    variance = values.var(ddof=1)
    mse = np.mean((values - target) ** 2)
    return {
        "estimator": name,
        "bias": bias,
        "variance": variance,
        "bias_squared": bias**2,
        "mse": mse,
        "variance_plus_bias_squared": variance + bias**2,
    }

pd.DataFrame(
    [
        summarize_estimator(estimator_unbiased, theta, "sample mean"),
        summarize_estimator(estimator_shrunken, theta, "shrink toward true theta"),
        summarize_estimator(estimator_wrong_shrinkage, theta, "shrink toward zero"),
    ]
)

,estimator,bias,variance,bias_squared,mse,variance_plus_bias_squared
0,sample mean,-0.0006,0.2004,0.0000,0.2004,0.2004
1,shrink toward true theta,-0.0006,0.1623,0.0000,0.1623,0.1623
2,shrink toward zero,-0.5006,0.1623,0.2506,0.4129,0.4129


## 4. Efficiency of estimators

If two estimators are both unbiased for the same parameter, the one with smaller variance is usually called **more efficient**. In regular parametric models, asymptotic efficiency is often expressed through the Cramer-Rao lower bound and Fisher information: an estimator is asymptotically efficient if its limiting variance reaches the best possible regular asymptotic variance.

A simple example is

$$
X_i \sim U[0, 2\theta].
$$

For this distribution, $E[X_i]=\theta$, so the sample mean $\overline X$ is unbiased for $\theta$. The population median is also $\theta$, so the sample median is a natural estimator too. Both are centered correctly, but they need not have the same variance.

The mean is more efficient under the exact uniform model. The median can be less efficient under the model, but more robust if observations are contaminated by outliers.

In [9]:
theta = 10.0
sample_sizes = np.array([5, 10, 20, 50, 100, 200])
trials = 50_000

efficiency_rows = []
for n in sample_sizes:
    data = RNG.uniform(0.0, 2.0 * theta, size=(trials, n))
    mean_estimator = data.mean(axis=1)
    median_estimator = np.median(data, axis=1)
    efficiency_rows.extend(
        [
            {
                "n": n,
                "estimator": "sample mean",
                "bias": mean_estimator.mean() - theta,
                "variance": mean_estimator.var(ddof=1),
                "mse": np.mean((mean_estimator - theta) ** 2),
            },
            {
                "n": n,
                "estimator": "sample median",
                "bias": median_estimator.mean() - theta,
                "variance": median_estimator.var(ddof=1),
                "mse": np.mean((median_estimator - theta) ** 2),
            },
        ]
    )

efficiency_frame = pd.DataFrame(efficiency_rows)
efficiency_frame

,n,estimator,bias,variance,mse
0,5,sample mean,0.0271,6.5683,6.5689
1,5,sample median,0.0327,14.1653,14.1661
2,10,sample mean,-0.0114,3.3168,3.3168
3,10,sample median,-0.0167,7.5565,7.5566
4,20,sample mean,0.0146,1.6776,1.6778
5,20,sample median,0.0287,4.3452,4.3459
6,50,sample mean,0.0016,0.6680,0.6680
7,50,sample median,0.0023,1.8763,1.8763
8,100,sample mean,-0.0005,0.3297,0.3297
9,100,sample median,-0.0034,0.9654,0.9654


In [10]:
fig = px.line(
    efficiency_frame,
    x="n",
    y="variance",
    color="estimator",
    markers=True,
    title="Efficiency comparison for U[0, 2 theta]",
    labels={"n": "sample size", "variance": "estimator variance"},
)
fig.update_xaxes(type="log")
fig.update_yaxes(type="log")
show_plot(fig)

## 5. Maximum likelihood estimation

Assume that the sample $X_1, \ldots, X_n$ comes from a parametric family with density or probability mass function $f(x; \theta)$, where $\theta \in \Theta$.

The **likelihood** of the observed sample is the joint density or probability of the data as a function of the parameter:

$$
L(\theta; X_1,\ldots,X_n)=\prod_{i=1}^n f(X_i;\theta).
$$

The **log-likelihood** is

$$
\ell(\theta)=\log L(\theta)=\sum_{i=1}^n \log f(X_i;\theta).
$$

The **maximum likelihood estimator (MLE)** is

$$
\widehat\theta_{ML}=\arg\max_{\theta \in \Theta} L(\theta)
=\arg\max_{\theta \in \Theta} \ell(\theta).
$$

The logarithm is monotone, so maximizing likelihood and log-likelihood gives the same parameter. We usually optimize the log-likelihood because products become sums and numerical underflow is reduced.

Under regularity conditions, MLEs have important large-sample properties:

- **Consistency:** $\widehat\theta_{ML} \xrightarrow{P} \theta_0$.
- **Invariance:** if $\widehat\theta$ estimates $\theta$, then $g(\widehat\theta)$ is the MLE for $g(\theta)$.
- **Asymptotic normality:** $\sqrt n(\widehat\theta_{ML}-\theta_0)$ is asymptotically normal.
- **Asymptotic efficiency:** its asymptotic variance reaches the Fisher-information bound in regular models.

These are asymptotic statements. They do not guarantee good behavior in every finite sample or every misspecified model.

### Bernoulli MLE

Let $X_i \sim Bernoulli(p)$. If $k=\sum_i X_i$, then

$$
L(p)=p^k(1-p)^{n-k}.
$$

The log-likelihood is

$$
\ell(p)=k\log p+(n-k)\log(1-p).
$$

Differentiate and set to zero:

$$
\frac{k}{p}-\frac{n-k}{1-p}=0.
$$

Solving gives

$$
\widehat p_{ML}=\frac{k}{n}=\overline X.
$$

In [33]:
bernoulli_sample = RNG.binomial(n=1, p=0.35, size=200)
p_mle = bernoulli_sample.mean()

pd.DataFrame(
    [
        {"quantity": "number of successes", "value": int(bernoulli_sample.sum())},
        {"quantity": "sample size", "value": int(bernoulli_sample.size)},
        {"quantity": "Bernoulli MLE p_hat", "value": p_mle},
    ]
)

,quantity,value
0,number of successes,73.0000
1,sample size,200.0000
2,Bernoulli MLE p_hat,0.3650


### Normal MLE

Let $X_i \sim N(\mu, \sigma^2)$. The log-likelihood is

$$
\ell(\mu,\sigma^2)
= -\frac{n}{2}\log(2\pi)
  -\frac{n}{2}\log\sigma^2
  -\frac{1}{2\sigma^2}\sum_{i=1}^n (X_i-\mu)^2.
$$

Maximizing over $\mu$ gives

$$
\widehat\mu_{ML}=\overline X.
$$

Plugging this into the likelihood and maximizing over $\sigma^2$ gives

$$
\widehat\sigma^2_{ML}
=\frac{1}{n}\sum_{i=1}^n (X_i-\overline X)^2.
$$

This variance estimator is biased downward:

$$
E[\widehat\sigma^2_{ML}]=\frac{n-1}{n}\sigma^2.
$$

The unbiased estimator divides by $n-1$, but the MLE remains consistent.

In [58]:
true_mu = 1.5
true_sigma = 2.0
sample = RNG.normal(loc=true_mu, scale=true_sigma, size=70)

mu_mle = sample.mean()
sigma2_mle = np.mean((sample - mu_mle) ** 2)
sigma2_unbiased = sample.var(ddof=1)

pd.DataFrame(
    [
        {"quantity": "true mu", "value": true_mu},
        {"quantity": "normal MLE mu_hat", "value": mu_mle},
        {"quantity": "true sigma^2", "value": true_sigma**2},
        {"quantity": "normal MLE sigma^2", "value": sigma2_mle},
        {"quantity": "unbiased sample variance", "value": sigma2_unbiased},
    ]
)

,quantity,value
0,true mu,1.5000
1,normal MLE mu_hat,1.6685
2,true sigma^2,4.0000
3,normal MLE sigma^2,4.2206
4,unbiased sample variance,4.2818


## 7. Logistic regression as Bernoulli MLE

For binary targets $y_i \in \{0,1\}$ and feature vectors $x_i$, logistic regression models

$$
P(y_i=1\mid x_i)=\sigma(\beta_0+x_i^T\beta),
$$

where

$$
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

Conditional on features, each target is Bernoulli with success probability $p_i=\sigma(\beta_0+x_i^T\beta)$. The likelihood is

$$
L(\beta_0,\beta)=\prod_{i=1}^n p_i^{y_i}(1-p_i)^{1-y_i}.
$$

The log-likelihood is

$$
\ell(\beta_0,\beta)=\sum_{i=1}^n
\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right].
$$

Maximizing this log-likelihood is equivalent to minimizing the negative log-likelihood, also called binary cross-entropy. Scikit-learn's `LogisticRegression` optimizes a regularized version of this objective by default.

### Bybit market data setup

We will fetch public hourly candles for `BTCUSDT`, `ETHUSDT`, and `SOLUSDT` from the Bybit V5 `/v5/market/kline` endpoint. The target is whether BTC's next-hour return is positive.

All features are built from current or past candles only, so the example avoids using future information in the feature matrix.

In [ ]:
BYBIT_KLINE_URL = "https://api.bybit.com/v5/market/kline"
CATEGORY = "linear"
INTERVAL = "60"
INTERVAL_MS = 60 * 60 * 1000
LIMIT = 1000

SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]
TARGET_SYMBOL = "BTCUSDT"
START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

KLINE_COLUMNS = [
    "start_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "turnover",
]


def utc_ms(date_text: str) -> int:
    """Convert a YYYY-MM-DD date to a UTC millisecond timestamp."""
    return int(pd.Timestamp(date_text, tz="UTC").timestamp() * 1000)


def request_kline_page(
    session: requests.Session,
    symbol: str,
    start_ms: int,
    end_ms: int,
) -> list[list[str]]:
    """Request one page of Bybit candles with light retry logic."""
    params: dict[str, Any] = {
        "category": CATEGORY,
        "symbol": symbol,
        "interval": INTERVAL,
        "start": start_ms,
        "end": end_ms,
        "limit": LIMIT,
    }

    for attempt in range(5):
        response = session.get(BYBIT_KLINE_URL, params=params, timeout=30)
        if response.status_code == 429:
            time.sleep(2**attempt)
            continue

        response.raise_for_status()
        payload = response.json()
        if payload.get("retCode") == 0:
            return payload["result"]["list"]

        if attempt == 4:
            raise RuntimeError(f"Bybit returned an error: {payload}")
        time.sleep(2**attempt)

    return []


def rows_to_frame(rows: list[list[str]], symbol: str) -> pd.DataFrame:
    """Convert raw Bybit kline rows into a typed pandas DataFrame."""
    frame = pd.DataFrame(rows, columns=KLINE_COLUMNS)
    for column in KLINE_COLUMNS:
        frame[column] = pd.to_numeric(frame[column])

    frame["timestamp"] = pd.to_datetime(frame["start_time"], unit="ms", utc=True)
    frame["symbol"] = symbol
    return frame[
        ["timestamp", "symbol", "open", "high", "low", "close", "volume", "turnover"]
    ]


def fetch_bybit_klines(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    """Fetch all hourly candles for one symbol in the requested date range."""
    start_ms = utc_ms(start_date)
    end_ms = utc_ms(end_date)
    cursor_end = end_ms
    all_rows: list[list[str]] = []

    with requests.Session() as session:
        while cursor_end >= start_ms:
            rows = request_kline_page(session, symbol, start_ms, cursor_end)
            if not rows:
                break

            all_rows.extend(rows)
            oldest_start = min(int(row[0]) for row in rows)
            if oldest_start <= start_ms:
                break

            cursor_end = oldest_start - INTERVAL_MS
            time.sleep(0.05)

    frame = rows_to_frame(all_rows, symbol)
    return (
        frame.drop_duplicates(subset=["timestamp"])
        .sort_values("timestamp")
        .query("@start_timestamp <= timestamp <= @end_timestamp")
        .reset_index(drop=True)
    )

In [60]:
frames = []
for symbol in SYMBOLS:
    frame = fetch_bybit_klines(symbol, START_DATE, END_DATE)
    print(
        f"{symbol}: {len(frame):,} candles from "
        f"{frame['timestamp'].min()} to {frame['timestamp'].max()}"
    )
    frames.append(frame)

candles = pd.concat(frames, ignore_index=True).sort_values(["symbol", "timestamp"])
candles.head()

BTCUSDT: 17,521 candles from 2024-01-01 00:00:00+00:00 to 2025-12-31 00:00:00+00:00
ETHUSDT: 17,521 candles from 2024-01-01 00:00:00+00:00 to 2025-12-31 00:00:00+00:00
SOLUSDT: 17,521 candles from 2024-01-01 00:00:00+00:00 to 2025-12-31 00:00:00+00:00


,timestamp,symbol,open,high,low,close,volume,turnover
0,2024-01-01 00:00:00+00:00,BTCUSDT,42324.8000,42610.9000,42300.2000,42517.4000,3038.2070,129022621.2567
1,2024-01-01 01:00:00+00:00,BTCUSDT,42517.4000,42842.9000,42475.1000,42661.3000,3308.2310,141296220.3691
2,2024-01-01 02:00:00+00:00,BTCUSDT,42661.3000,42691.9000,42550.8000,42631.8000,1651.4470,70384296.4473
3,2024-01-01 03:00:00+00:00,BTCUSDT,42631.8000,42641.4000,42271.5000,42384.1000,3499.2770,148488084.3470
4,2024-01-01 04:00:00+00:00,BTCUSDT,42384.1000,42448.0000,42250.5000,42446.3000,1800.1570,76253111.4863


In [61]:
def build_symbol_features(frame: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """Create current-and-past features for one symbol."""
    working = frame.sort_values("timestamp").copy()
    close = working["close"]

    features = pd.DataFrame({"timestamp": working["timestamp"]})
    features[f"{prefix}_return_1h"] = close.pct_change(1)
    features[f"{prefix}_return_3h"] = close.pct_change(3)
    features[f"{prefix}_return_12h"] = close.pct_change(12)
    features[f"{prefix}_rolling_mean_24h"] = close.pct_change().rolling(24).mean()
    features[f"{prefix}_volatility_24h"] = close.pct_change().rolling(24).std()
    features[f"{prefix}_candle_range_pct"] = (working["high"] - working["low"]) / close
    features[f"{prefix}_volume_change_1h"] = working["volume"].pct_change(1)
    features[f"{prefix}_turnover_change_1h"] = working["turnover"].pct_change(1)
    return features


symbol_prefixes = {
    "BTCUSDT": "btc",
    "ETHUSDT": "eth",
    "SOLUSDT": "sol",
}

feature_frames = []
for symbol, prefix in symbol_prefixes.items():
    symbol_frame = candles.query("symbol == @symbol")
    feature_frames.append(build_symbol_features(symbol_frame, prefix))

features = feature_frames[0]
for feature_frame in feature_frames[1:]:
    features = features.merge(feature_frame, on="timestamp", how="inner")

btc = candles.query("symbol == @TARGET_SYMBOL").sort_values("timestamp").copy()
btc["btc_return_next_hour"] = btc["close"].pct_change().shift(-1)
target = btc[["timestamp", "btc_return_next_hour"]]

model_frame = features.merge(target, on="timestamp", how="inner")
model_frame["target"] = (model_frame["btc_return_next_hour"] > 0).astype(int)
model_frame["btc_minus_eth_return_1h"] = (
    model_frame["btc_return_1h"] - model_frame["eth_return_1h"]
)
model_frame["btc_minus_sol_return_1h"] = (
    model_frame["btc_return_1h"] - model_frame["sol_return_1h"]
)

feature_columns = [
    "btc_return_1h",
    "btc_return_3h",
    "btc_return_12h",
    "btc_rolling_mean_24h",
    "btc_volatility_24h",
    "btc_candle_range_pct",
    "btc_volume_change_1h",
    "btc_turnover_change_1h",
    "eth_return_1h",
    "eth_return_12h",
    "sol_return_1h",
    "sol_return_12h",
    "btc_minus_eth_return_1h",
    "btc_minus_sol_return_1h",
]

model_frame = (
    model_frame.replace([np.inf, -np.inf], np.nan)
    .dropna(subset=feature_columns + ["target"])
    .reset_index(drop=True)
)

model_frame[["timestamp", "target", "btc_return_next_hour", *feature_columns]].head()

,timestamp,target,btc_return_next_hour,btc_return_1h,btc_return_3h,btc_return_12h,btc_rolling_mean_24h,btc_volatility_24h,btc_candle_range_pct,btc_volume_change_1h,btc_turnover_change_1h,eth_return_1h,eth_return_12h,sol_return_1h,sol_return_12h,btc_minus_eth_return_1h,btc_minus_sol_return_1h
0,2024-01-02 00:00:00+00:00,0,-0.0057,0.0215,0.0346,0.0585,0.0026,0.0060,0.0258,0.9112,0.9433,0.0161,0.0392,0.0063,0.0563,0.0055,0.0152
1,2024-01-02 01:00:00+00:00,1,0.0111,-0.0057,0.0310,0.0507,0.0022,0.0062,0.0167,-0.5228,-0.5191,-0.0044,0.0325,0.0047,0.0560,-0.0013,-0.0104
2,2024-01-02 02:00:00+00:00,1,0.0001,0.0111,0.0270,0.0652,0.0027,0.0064,0.0112,-0.4345,-0.4333,0.0014,0.0355,0.0068,0.0723,0.0098,0.0044
3,2024-01-02 03:00:00+00:00,0,-0.0047,0.0001,0.0054,0.0609,0.0029,0.0062,0.0099,0.4531,0.4586,0.0013,0.0306,-0.0045,0.0518,-0.0013,0.0046
4,2024-01-02 04:00:00+00:00,1,0.0000,-0.0047,0.0065,0.0572,0.0027,0.0064,0.0063,-0.4814,-0.4818,0.0004,0.0331,0.0092,0.0547,-0.0051,-0.0139


In [62]:
split_index = int(len(model_frame) * 0.80)
train_frame = model_frame.iloc[:split_index].copy()
test_frame = model_frame.iloc[split_index:].copy()

X_train = train_frame[feature_columns]
y_train = train_frame["target"]
X_test = test_frame[feature_columns]
y_test = test_frame["target"]

classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE),
)
classifier.fit(X_train, y_train)

test_probabilities = classifier.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= 0.5).astype(int)

metrics = pd.DataFrame(
    [
        {
            "accuracy": accuracy_score(y_test, test_predictions),
            "balanced_accuracy": balanced_accuracy_score(y_test, test_predictions),
            "roc_auc": roc_auc_score(y_test, test_probabilities),
            "positive_rate_train": y_train.mean(),
            "positive_rate_test": y_test.mean(),
        }
    ]
)
metrics

,accuracy,balanced_accuracy,roc_auc,positive_rate_train,positive_rate_test
0,0.5103,0.5098,0.5231,0.5078,0.5020


In [63]:
prediction_frame = test_frame[["timestamp", "btc_return_next_hour", "target"]].copy()
prediction_frame["predicted_probability"] = test_probabilities
prediction_frame["prediction"] = test_predictions
prediction_frame.head(10)

,timestamp,btc_return_next_hour,target,predicted_probability,prediction
13997,2025-08-07 05:00:00+00:00,0.0007,1,0.4825,0
13998,2025-08-07 06:00:00+00:00,0.0013,1,0.4869,0
13999,2025-08-07 07:00:00+00:00,0.0014,1,0.4880,0
14000,2025-08-07 08:00:00+00:00,0.0009,1,0.4714,0
14001,2025-08-07 09:00:00+00:00,0.0110,1,0.4814,0
14002,2025-08-07 10:00:00+00:00,0.0014,1,0.3757,0
14003,2025-08-07 11:00:00+00:00,-0.0011,0,0.4626,0
14004,2025-08-07 12:00:00+00:00,0.0023,1,0.4646,0
14005,2025-08-07 13:00:00+00:00,-0.0024,0,0.4649,0
14006,2025-08-07 14:00:00+00:00,0.0027,1,0.5066,1


In [64]:
confusion = confusion_matrix(y_test, test_predictions, labels=[0, 1])
confusion_frame = pd.DataFrame(
    confusion,
    index=["actual down-or-flat", "actual up"],
    columns=["predicted down-or-flat", "predicted up"],
)
confusion_frame

,predicted down-or-flat,predicted up
actual down-or-flat,683,1060
actual up,654,1103


### ROC curve and ROC-AUC

A **ROC curve** shows the tradeoff between true positive rate and false positive rate across all classification thresholds. Instead of fixing the cutoff at `0.5`, we move the threshold from very high to very low and track the ranking behavior of model scores.

The **true positive rate** is

$$
TPR = \frac{TP}{TP+FN}.
$$

The **false positive rate** is

$$
FPR = \frac{FP}{FP+TN}.
$$

**ROC-AUC** is the area under the ROC curve. It can be interpreted as the probability that the model assigns a higher score to a randomly chosen positive example than to a randomly chosen negative example.

`ROC-AUC = 0.5` corresponds to random ranking, `ROC-AUC = 1.0` is perfect ranking, and values below `0.5` mean the ranking is worse than random.

In [65]:
fpr, tpr, thresholds = roc_curve(y_test, test_probabilities)
auc_value = roc_auc_score(y_test, test_probabilities)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        line={"width": 3},
        name=f"logistic regression ROC-AUC={auc_value:.3f}",
        hovertemplate="FPR=%{x:.3f}<br>TPR=%{y:.3f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        line={"color": "black", "dash": "dash"},
        name="random ranking",
        hoverinfo="skip",
    )
)
fig.update_layout(
    title="ROC curve for next-hour BTC direction",
    xaxis_title="false positive rate",
    yaxis_title="true positive rate",
    template="plotly_white",
)
fig.update_xaxes(range=[0, 1])
fig.update_yaxes(range=[0, 1])
show_plot(fig)

In [66]:
logistic_model = classifier.named_steps["logisticregression"]
coefficient_frame = pd.DataFrame(
    {
        "feature": feature_columns,
        "coefficient": logistic_model.coef_[0],
    }
).sort_values("coefficient")

fig = px.bar(
    coefficient_frame,
    x="coefficient",
    y="feature",
    orientation="h",
    title="Standardized logistic regression coefficients",
    labels={"coefficient": "coefficient", "feature": "feature"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
show_plot(fig, height=620)

The coefficient chart is easiest to read because the model is fit after standardization. A positive coefficient increases the log-odds of a positive next-hour BTC return when that feature increases by one training-set standard deviation, holding other features fixed. A negative coefficient decreases those log-odds.

The important statistical connection is this: logistic regression estimates parameters by maximizing the conditional Bernoulli likelihood of the observed labels. The trading-looking target is only a concrete binary response; the underlying estimation principle is the same MLE principle used earlier in the notebook.